# Batch Normalization in Deep Learning — Beginner Masterclass

**Google Colab-compatible notebook based on:** [Batch Normalization in Deep Learning — GeeksforGeeks](https://www.geeksforgeeks.org/deep-learning/batch-normalization-in-deep-learning-2/)

This notebook teaches Batch Normalization from first principles, uses small numerical examples, visualizes mini-batch statistics, and trains a small neural network on MNIST.

## What you will learn

1. Why Batch Normalization is used.
2. How to calculate mini-batch mean and variance by hand.
3. How normalization, gamma, and beta work.
4. Why Batch Normalization is normally placed before the activation in a Dense network.
5. The difference between training and inference.
6. How to add Batch Normalization in TensorFlow/Keras and PyTorch.
7. How to compare a network with Batch Normalization against one without it.
8. Benefits, limitations, applications, and common mistakes.

**Source note:** This notebook summarizes the article in original wording and adds explanatory calculations and experiments; it does not reproduce the article verbatim. The original article is linked above for reference.

## 1. What is Batch Normalization?

Batch Normalization (often abbreviated **BN**) is a layer used inside a neural network. During training, it normalizes activations using statistics computed from the current mini-batch, then applies a learnable scale and shift.

The article highlights several reasons to use it:

- It can make optimization faster or more stable.
- It can make training more tolerant of larger learning rates.
- It can improve gradient flow in some deep networks.
- Variation in mini-batch statistics can have a regularizing effect.
- It can make training less sensitive to weight initialization.

These are common benefits, **not guarantees**. Whether Batch Normalization helps depends on the architecture, batch size, optimizer, learning rate, data, and training setup.

### A useful distinction

**Input normalization** (such as dividing image pixels by 255) preprocesses the original dataset.

**Batch Normalization** is a trainable layer inside the model that normalizes intermediate activations using mini-batch statistics during training.

They are related ideas, but they are not the same operation.

## 2. Why might a network benefit from Batch Normalization?

Imagine a layer receives activations with a mean of 80 and a standard deviation of 25. During training, updates to earlier layers may shift the scale and distribution of these activations.

Batch Normalization standardizes activations using the current mini-batch's mean and variance. It can make the numerical scale seen by later layers more predictable during training.

The article describes this in terms of reducing *internal covariate shift*. More broadly, Batch Normalization changes the optimization landscape and the way gradients behave; it is best not to assume that only one mechanism explains all of its benefits.

**Visual intuition:** if the batch activations are \([2,4,6,8]\), subtracting their mean centers the values around zero. Dividing by their standard deviation then scales their spread.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

batch = np.array([2.0, 4.0, 6.0, 8.0])
batch_mean = batch.mean()
batch_var = np.mean((batch - batch_mean) ** 2)
batch_std = np.sqrt(batch_var)
eps = 1e-3

batch_normalized = (batch - batch_mean) / np.sqrt(batch_var + eps)

print("Batch values:", batch)
print("Mean:", batch_mean)
print("Variance (population formula):", batch_var)
print("Standard deviation:", batch_std)
print("Normalized values:", batch_normalized)
print("Normalized mean:", batch_normalized.mean())
print("Normalized variance:", batch_normalized.var())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(len(batch)), batch)
axes[0].set_title("Original mini-batch")
axes[0].set_xlabel("Sample")
axes[0].set_ylabel("Activation")
axes[0].set_xticks(range(len(batch)))

axes[1].bar(range(len(batch)), batch_normalized)
axes[1].axhline(0, linewidth=1)
axes[1].set_title("After Batch Normalization (before gamma/beta)")
axes[1].set_xlabel("Sample")
axes[1].set_ylabel("Normalized activation")
axes[1].set_xticks(range(len(batch)))

plt.tight_layout()
plt.show()

## 3. The Batch Normalization formula

For one activation feature and a mini-batch with \(m\) samples:

### Step 1 — Calculate the batch mean

\[
\mu_B=\frac{1}{m}\sum_{i=1}^{m}x_i
\]

### Step 2 — Calculate the batch variance

\[
\sigma_B^2=\frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_B)^2
\]

This uses the population-style variance with denominator \(m\), matching the usual Batch Normalization definition.

### Step 3 — Normalize

\[
\hat{x}_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
\]

The small positive constant \(\epsilon\) helps prevent division by zero and improves numerical stability.

### Step 4 — Learn a scale and shift

\[
\boxed{y_i=\gamma\hat{x}_i+\beta}
\]

- \(\gamma\) (gamma) is a learned scale parameter.
- \(\beta\) (beta) is a learned shift parameter.
- \(\hat{x}_i\) is the normalized activation.
- \(y_i\) is the output passed to the next layer.

The normalized values have approximately zero mean and unit variance before the learned scale and shift, assuming a non-degenerate batch and ignoring small numerical effects from \(\epsilon\).

## 4. Work through Batch Normalization by hand

Use this mini-batch:

\[
x=[2,4,6,8]
\]

First, calculate the mean:

\[
\mu_B=(2+4+6+8)/4=5
\]

Next, calculate the variance:

\[
\sigma_B^2=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}
\]

\[
=\frac{9+1+1+9}{4}=5
\]

Then normalize:

\[
\hat{x}_i=\frac{x_i-5}{\sqrt{5+\epsilon}}
\]

Finally, with \(\gamma=2\) and \(\beta=1\):

\[
y_i=2\hat{x}_i+1
\]

At initialization, frameworks commonly use \(\gamma=1\) and \(\beta=0\), so the layer starts by approximately standardizing the activations.

In [ ]:
x = np.array([2.0, 4.0, 6.0, 8.0])
m = len(x)
mu = np.mean(x)
var = np.mean((x - mu) ** 2)
epsilon = 1e-3
x_hat = (x - mu) / np.sqrt(var + epsilon)

gamma = 2.0
beta = 1.0
y = gamma * x_hat + beta

manual_table = pd.DataFrame({
    "x": x,
    "x - batch mean": x - mu,
    "(x - mean)^2": (x - mu) ** 2,
    "normalized x_hat": x_hat,
    "output gamma*x_hat + beta": y
})

print(f"m = {m}")
print(f"Batch mean = {mu:.4f}")
print(f"Batch variance = {var:.4f}")
print(f"epsilon = {epsilon}")
print(f"gamma = {gamma}, beta = {beta}")
display(manual_table.round(4))

## 5. Why are gamma and beta trainable?

If Batch Normalization only standardized values and stopped there, it could constrain what the following layer receives. The trainable parameters let the network learn the scale and offset that work best.

\[
y=\gamma\hat{x}+\beta
\]

Try these settings:

- \(\gamma=1,\beta=0\): keep normalized values.
- \(\gamma=2,\beta=0\): double their scale.
- \(\gamma=1,\beta=3\): shift them upward by 3.
- \(\gamma=0.5,\beta=-1\): shrink them and shift them downward.

Gamma and beta are learned through backpropagation. They are distinct from the running mean and variance, which are stored for inference.

In [ ]:
x = np.array([2.0, 4.0, 6.0, 8.0])
mu = x.mean()
var = np.mean((x - mu) ** 2)
x_hat = (x - mu) / np.sqrt(var + 1e-3)

settings = [
    (1.0, 0.0, "gamma=1, beta=0"),
    (2.0, 0.0, "gamma=2, beta=0"),
    (1.0, 3.0, "gamma=1, beta=3"),
    (0.5, -1.0, "gamma=0.5, beta=-1"),
]

plt.figure(figsize=(9, 4))
for gamma, beta, label in settings:
    plt.plot(x_hat, gamma * x_hat + beta, marker="o", label=label)
plt.axhline(0, linewidth=0.8)
plt.title("Learnable scale and shift after normalization")
plt.xlabel("Normalized activation x_hat")
plt.ylabel("BatchNorm output")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Batch Normalization in a neural network

A common ordering in a feed-forward network is:

\[
\text{Dense} \rightarrow \text{BatchNorm} \rightarrow \text{ReLU}
\]

The Dense layer calculates a weighted sum. Batch Normalization standardizes that layer's activations across the mini-batch. ReLU then applies the nonlinear activation.

A final classifier might look like:

\[
\text{Dense} \rightarrow \text{Softmax}
\]

Batch Normalization is not usually needed on the final probability output. It is commonly inserted in hidden layers.

### Training versus inference

- **Training:** use the current mini-batch's mean and variance; update running estimates.
- **Inference / evaluation:** use the running estimates collected during training, not statistics computed from each new test batch.

This matters because predictions should be stable when examples are served individually or in different batch groupings.

## 7. Training mode versus inference mode — conceptual simulation

We'll make two groups with different means. Batch Normalization applied in training mode normalizes each mini-batch using its own statistics. That means the normalized values of an example may differ depending on which other examples share its batch.

During inference, frameworks generally use the running mean and variance estimated during training.

This is why the same layer needs distinct training and inference behaviour.

In [ ]:
batch_a = np.array([1.0, 2.0, 3.0, 4.0])
batch_b = np.array([11.0, 12.0, 13.0, 14.0])

def normalize_batch(values, epsilon=1e-3):
    return (values - values.mean()) / np.sqrt(values.var() + epsilon)

print("Batch A raw:", batch_a)
print("Batch A normalized during training:", normalize_batch(batch_a))
print()
print("Batch B raw:", batch_b)
print("Batch B normalized during training:", normalize_batch(batch_b))
print()
print("The same relative pattern is normalized around each batch's own mean.")

## 8. Load and prepare MNIST

The article uses MNIST, a dataset of 28×28 grayscale handwritten digit images labelled 0–9.

We will:
1. Load MNIST with TensorFlow/Keras.
2. Scale pixels from integers in \([0,255]\) to floating-point values in \([0,1]\).
3. Flatten each 28×28 image to a vector of 784 values for a Dense network.
4. Use a subset of 5,000 training images so the demonstration runs faster.
5. Keep a test subset for evaluation.

**Pixel scaling is input preprocessing. Batch Normalization is a separate layer applied to hidden activations.**

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.mnist.load_data()

# Input preprocessing: pixel scale [0, 255] -> [0, 1]
x_train_full = x_train_full.astype("float32") / 255.0
x_test_full = x_test_full.astype("float32") / 255.0

# Flatten each 28 x 28 image into a 784-element vector.
x_train_full = x_train_full.reshape(-1, 784)
x_test_full = x_test_full.reshape(-1, 784)

# Small subsets to keep this beginner demo faster.
x_train = x_train_full[:5000]
y_train = y_train_full[:5000]
x_test = x_test_full[:2000]
y_test = y_test_full[:2000]

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("Pixel range:", x_train.min(), "to", x_train.max())

### Inspect a few MNIST examples

Before building the network, view a few images and their labels. Even though the model receives flattened arrays, displaying the original 28×28 pixels helps connect the numerical input to the real task.

In [ ]:
plt.figure(figsize=(9, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.suptitle("Sample MNIST digits")
plt.tight_layout()
plt.show()

## 9. Build the Keras model with Batch Normalization

The article demonstrates this sequence:

1. Input with 784 features.
2. Dense layer with 64 neurons.
3. Batch Normalization.
4. ReLU activation.
5. Dense output layer with 10 softmax classes.

We will add a second hidden Dense block for a slightly richer demonstration, while preserving the same ordering.

In [ ]:
def build_keras_model(use_batch_norm=True):
    model_layers = [
        keras.Input(shape=(784,)),
        layers.Dense(128, use_bias=not use_batch_norm)
    ]

    if use_batch_norm:
        model_layers.append(layers.BatchNormalization())

    model_layers += [
        layers.Activation("relu"),
        layers.Dense(64, use_bias=not use_batch_norm)
    ]

    if use_batch_norm:
        model_layers.append(layers.BatchNormalization())

    model_layers += [
        layers.Activation("relu"),
        layers.Dense(10, activation="softmax")
    ]

    model = keras.Sequential(model_layers)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model_bn = build_keras_model(use_batch_norm=True)
model_bn.summary()

## 10. Train the model with Batch Normalization

The article trains for two epochs with mini-batches of 32 examples. We'll use a similar short run.

The labels are integers from 0 to 9, so `sparse_categorical_crossentropy` is appropriate. The output layer uses softmax to produce 10 class probabilities.

Training time and final accuracy will vary by runtime, hardware, and framework version. Treat printed metrics as your run's results, not as universal benchmarks.

In [ ]:
EPOCHS = 2
BATCH_SIZE = 32

history_bn = model_bn.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history_bn.history["loss"], marker="o", label="Train")
plt.plot(history_bn.history["val_loss"], marker="o", label="Validation")
plt.title("BatchNorm model: loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.grid(alpha=0.25)

plt.subplot(1, 2, 2)
plt.plot(history_bn.history["accuracy"], marker="o", label="Train")
plt.plot(history_bn.history["val_accuracy"], marker="o", label="Validation")
plt.title("BatchNorm model: accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 11. Compare with a similar model without Batch Normalization

Now we train the same general architecture without Batch Normalization. This is a controlled educational comparison, but it is not proof that BN will always outperform a model without BN.

To make the architecture comparison clearer, Dense layers use biases when BN is absent. When BN is present, Dense biases are disabled because the subsequent Batch Normalization layer has a trainable shift parameter \(\beta\); this is a common design choice.

We use the same data subset, batch size, optimizer, learning rate, and epoch count. Because initial weights and minibatch order can still influence results, differences may vary across runs.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
model_no_bn = build_keras_model(use_batch_norm=False)

history_no_bn = model_no_bn.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "With Batch Normalization",
        "Final train accuracy": history_bn.history["accuracy"][-1],
        "Final validation accuracy": history_bn.history["val_accuracy"][-1],
        "Final train loss": history_bn.history["loss"][-1],
        "Final validation loss": history_bn.history["val_loss"][-1],
    },
    {
        "Model": "Without Batch Normalization",
        "Final train accuracy": history_no_bn.history["accuracy"][-1],
        "Final validation accuracy": history_no_bn.history["val_accuracy"][-1],
        "Final train loss": history_no_bn.history["loss"][-1],
        "Final validation loss": history_no_bn.history["val_loss"][-1],
    }
])
display(comparison.round(4))

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history_bn.history["loss"], marker="o", label="With BN")
plt.plot(history_no_bn.history["loss"], marker="o", label="Without BN")
plt.title("Training loss comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.25)

plt.subplot(1, 2, 2)
plt.plot(history_bn.history["val_accuracy"], marker="o", label="With BN")
plt.plot(history_no_bn.history["val_accuracy"], marker="o", label="Without BN")
plt.title("Validation accuracy comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 12. Inspect Batch Normalization's learned parameters and moving statistics

Keras Batch Normalization contains:

- `gamma`: trainable scale values.
- `beta`: trainable shift values.
- `moving_mean`: running estimate of the feature means.
- `moving_variance`: running estimate of the feature variances.

Gamma and beta are trainable parameters. The moving mean and variance are updated during training but are not learned by gradient descent in the same way as gamma and beta.

Let's inspect the first Batch Normalization layer after training.

In [ ]:
bn_layers = [layer for layer in model_bn.layers if isinstance(layer, layers.BatchNormalization)]

if bn_layers:
    first_bn = bn_layers[0]
    gamma_values = first_bn.gamma.numpy()
    beta_values = first_bn.beta.numpy()
    moving_mean = first_bn.moving_mean.numpy()
    moving_var = first_bn.moving_variance.numpy()

    print("Number of normalized features:", len(gamma_values))
    print("First 10 gamma values:", gamma_values[:10])
    print("First 10 beta values:", beta_values[:10])
    print("First 10 moving means:", moving_mean[:10])
    print("First 10 moving variances:", moving_var[:10])

    param_table = pd.DataFrame({
        "gamma": gamma_values[:10],
        "beta": beta_values[:10],
        "moving_mean": moving_mean[:10],
        "moving_variance": moving_var[:10]
    })
    display(param_table.round(4))
else:
    print("No BatchNormalization layer found.")

## 13. Observe training mode versus inference mode in Keras

Keras uses `training=True` to normalize using the current mini-batch statistics and update the running estimates. With `training=False`, it uses the stored moving estimates.

The same data can therefore produce slightly different activations when the model is explicitly run in training mode versus inference mode, particularly if the running statistics are still changing or the chosen batch differs from the running distribution.

In [ ]:
bn_probe = keras.Model(
    inputs=model_bn.input,
    outputs=bn_layers[0].output
)

probe_batch = x_train[:32]
out_training = bn_probe(probe_batch, training=True).numpy()
out_inference = bn_probe(probe_batch, training=False).numpy()

print("Output mean with training=True:", out_training.mean(axis=0)[:8])
print("Output variance with training=True:", out_training.var(axis=0)[:8])
print("Output mean with training=False:", out_inference.mean(axis=0)[:8])
print("Output variance with training=False:", out_inference.var(axis=0)[:8])

print("\\nMean absolute difference between outputs:",
      np.mean(np.abs(out_training - out_inference)))

## 14. PyTorch implementation based on the article

The article also demonstrates Batch Normalization with PyTorch. In PyTorch, the equivalent layer for a dense feature vector is `nn.BatchNorm1d`.

The typical order is:

```python
linear -> BatchNorm1d -> ReLU -> next layer
```

For image batches with shape `(batch, channels, height, width)`, the common layer is `nn.BatchNorm2d(channels)`.

The following is a complete compact PyTorch training example using MNIST. It is optional because the TensorFlow experiment above already trains and compares two models.

In [ ]:
# Optional PyTorch demo:
# Set RUN_PYTORCH_DEMO = True to train this model after the TensorFlow cells.
RUN_PYTORCH_DEMO = False

if RUN_PYTORCH_DEMO:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset

    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    x_pt = torch.tensor(x_train.reshape(-1, 1, 28, 28), dtype=torch.float32)
    y_pt = torch.tensor(y_train, dtype=torch.long)
    train_ds = TensorDataset(x_pt, y_pt)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

    class MNISTBatchNormModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.flatten = nn.Flatten()
            self.fc1 = nn.Linear(784, 64, bias=False)
            self.bn1 = nn.BatchNorm1d(64)
            self.relu1 = nn.ReLU()
            self.fc2 = nn.Linear(64, 10)

        def forward(self, x):
            x = self.flatten(x)
            x = self.fc1(x)
            x = self.bn1(x)
            x = self.relu1(x)
            return self.fc2(x)  # raw logits for CrossEntropyLoss

    torch_model = MNISTBatchNormModel().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(torch_model.parameters(), lr=0.001)

    for epoch in range(2):
        torch_model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = torch_model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch + 1}, average batch loss: {running_loss / len(train_loader):.4f}")

    torch_model.eval()
    with torch.no_grad():
        test_x_pt = torch.tensor(
            x_test.reshape(-1, 1, 28, 28), dtype=torch.float32
        ).to(device)
        test_y_pt = torch.tensor(y_test, dtype=torch.long).to(device)
        predictions = torch_model(test_x_pt).argmax(dim=1)
        test_accuracy = (predictions == test_y_pt).float().mean().item()

    print(f"PyTorch test accuracy on the selected subset: {test_accuracy * 100:.2f}%")
else:
    print("PyTorch demo skipped. Set RUN_PYTORCH_DEMO = True to run it.")

## 15. Applications, advantages, and limitations

### Applications mentioned in the article

- **Image classification:** can help stabilize training in CNN architectures such as ResNet and VGG.
- **Object detection:** used in networks such as Faster R-CNN and YOLO.
- **Medical image analysis:** useful in training models for disease detection and image segmentation.
- **Speech recognition:** can be used to normalize internal representations of audio features.
- **Transfer learning:** often appears in pretrained architectures that are fine-tuned for new tasks.

These are examples of where Batch Normalization is used, not a guarantee of improved results in every model.

### Advantages

- May improve optimization speed.
- May allow larger learning rates in some models.
- Can help maintain more predictable activation scales.
- Can reduce sensitivity to the initial weight scale.
- Mini-batch variation can sometimes provide a regularization effect.

### Limitations

- Very small batches can give noisy statistics.
- Computing statistics and maintaining running estimates adds overhead.
- Training and inference use different statistics pathways.
- Batch-dependent normalization can be less convenient for sequence models or workloads with small/variable batch sizes. Layer Normalization is often used in Transformers and many recurrent architectures.

**Remember:** Batch Normalization does not replace sensible initialization, good preprocessing, a suitable learning rate, or proper evaluation.

## 16. Quick quiz

Answer these before revealing the suggested answers.

1. Why does Batch Normalization calculate a mean and variance?
2. What is the role of epsilon?
3. What do gamma and beta learn?
4. What is the difference between training and inference?
5. Why should Batch Normalization not be treated as identical to scaling input pixels by 255?
6. Why can very small mini-batches be a problem?
7. Which PyTorch layer is commonly used for Batch Normalization on dense feature vectors?

In [ ]:
answers = {
    "1": "It estimates the current mini-batch activation distribution so the values can be centered and scaled.",
    "2": "It improves numerical stability and prevents division by zero when variance is zero or extremely small.",
    "3": "Gamma is a trainable scale; beta is a trainable shift.",
    "4": "Training uses current mini-batch statistics and updates running estimates; inference generally uses stored running statistics.",
    "5": "Pixel scaling preprocesses raw input values; BatchNorm is a model layer applied to internal activations.",
    "6": "Small batches can produce noisy mean and variance estimates.",
    "7": "torch.nn.BatchNorm1d."
}

for question, answer in answers.items():
    print(f"Q{question}: {answer}")

## 17. Final recap

Batch Normalization transforms an activation using:

\[
\mu_B=\frac{1}{m}\sum_i x_i
\]

\[
\sigma_B^2=\frac{1}{m}\sum_i(x_i-\mu_B)^2
\]

\[
\hat{x}_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
\]

\[
\boxed{y_i=\gamma\hat{x}_i+\beta}
\]

During training it uses mini-batch statistics and updates running estimates. During inference it normally uses the running estimates collected during training.

The article demonstrates Batch Normalization on MNIST with both TensorFlow/Keras and PyTorch, and discusses its use cases, benefits, and limitations. This notebook expands those ideas with a by-hand calculation, visualizations, an executed Keras comparison, inspection of learned/running parameters, and an optional PyTorch experiment.

### Source

[GeeksforGeeks — Batch Normalization in Deep Learning](https://www.geeksforgeeks.org/deep-learning/batch-normalization-in-deep-learning-2/)

**Try next:** change the batch size, number of epochs, or hidden-layer width and observe how the curves and Batch Normalization statistics change.